# Modeling: peat condition -> fire

End-to-end scaffold for the modeling phase (see `modeling_roadmap.md`,
`decisions.md`). Flow:

1. Load the peat frame (80% histosol), restoration sites (treatment), covariates.
2. **Build the treated/control pixel-year panel** -- calendar-year dimension,
   per-year `treated`, and `years_after_treatment` event time.
   - **2b.** Thread the restoration year into `did.attach_cohort` and fit the
     staggered difference-in-differences (Callaway & Sant'Anna / Castro et al.).
3. `build_frame` -> tidy pixel-year table with a swappable fire response
   (the **levels / odds-ratio** route; consumes matched `units` from Stage 5-7).
4. `fit_logit_clustered` -> `odds_ratios`.

Only elevation + histosol % are on disk today, so we match/adjust on those two
now and add distance-to-coast / land cover once they download.

In [ ]:
# standard library
from pathlib import Path

# third-party
import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr

# local (peatfire)
from peatfire import (
    data_path,
    build_common_grid,
    to_common_grid,
    rasterize_polygons_to_grid,
    load_standardized,
)
from peatfire.modeling import (
    load_completed_restoration_sites_in_analysis_crs,
    available_covariates,
    covariate_on_grid,
    build_frame,
    fit_logit_clustered,
    odds_ratios,
    pixelate,
)
from peatfire.modeling import did
from peatfire.modeling.matching import get_treated_and_control_pixels, attach_covariates


print("covariates on disk:", available_covariates())

## 1. Load treatment + peat frame

In [3]:
# Restoration polygons = the TREATMENT (reprojected to EPSG:5070; metres).
restoration_yr_col = 'End_Yr'
peat_restoration = load_completed_restoration_sites_in_analysis_crs(restoration_yr_col=restoration_yr_col) # drops rows with 0 for the restoration_yr_col and reprojects to EPSG:5070


In [4]:
# 80% peat extent as the sample frame.
aoi_nc_peat_80_histosol = gpd.read_file(data_path("processed", "peat_extent", "nc_peatlands_80_histosol_aoi.gpkg"))

## 2. Build matched controls

**a. Clip-to-drop** the restoration sites (plus a buffer) from the peat frame to
get the control *candidate* pool. `how="difference"` is the inverse of clip:

```python
BUFFER_M = 1000  # exclusion halo around restoration sites, in metres
exclusion = gpd.GeoDataFrame(geometry=[restoration.buffer(BUFFER_M).union_all()],
                             crs=restoration.crs)
candidates = gpd.overlay(aoi_nc_peat_80_histosol, exclusion, how="difference")
```

**b. Candidate pixels + nearest site.** Turn candidate peat into pixel points
(e.g. centroids of the grid cells over `candidates`), then attach each one's
nearest restoration site and distance in one call:

```python
cand_pts = gpd.sjoin_nearest(candidate_points, restoration,
                             how="left", distance_col="dist_m")
```

**c. Sample covariates** at treated and candidate pixels (reuse the grid warp):
`covariate_on_grid('elevation', grid, aoi)` and `('histosol_pct', ...)`, then
read values at the pixel locations. Assemble a table with a `treatment` 0/1
column + `elevation`, `histosol_pct`.

**d. Match** treated<->control on those covariates. Two options:
- literal nearest-neighbour on z-scored covariates: `sklearn.neighbors.NearestNeighbors`
- propensity score: `pymatch.Matcher(...).fit_scores(); .match(); .matched_data`
  (matches on P(treatment), not raw covariates -- know the difference).

Output of this section: a `units` GeoDataFrame with **`treated`** (1/0),
**`site_id`** (the matched stratum), and **`unit_id`** columns. That is all
`build_frame` needs.

In [ ]:
spillover_m = 1000
res_m = 300
years = range(2019, 2025)          # FireCCIS311 coverage
covariates = ['histosol_pct', 'elevation']

# Pixel-YEAR panel (not a single flattened set): every candidate pixel is
# repeated once per calendar year, with
#   - `treated`               1 once the pixel's site has been restored, else 0
#   - `restoration_year`      the site's End_Yr (NaN for control-pool pixels)
#   - `years_after_treatment` = year - restoration_year  (0 = restoration year,
#                               matching the notebook's event_year convention)
# Control pixels appear in every year with treated=0. Pre-restoration site
# pixels are kept as *not-yet-treated* controls for the staggered DiD below
# (pass drop_pretreatment=True for the simpler restored-by-year-Y cross-section).
pixels = get_treated_and_control_pixels(
    aoi_nc_peat_80_histosol,
    peat_restoration,
    years=years,
    spillover_m=spillover_m,
    res_m=res_m,
    treated_col_name="treated",
    restoration_yr_col=restoration_yr_col,   # 'End_Yr'
    site_col="Proj_Name",
)

# Static covariates are sampled once per pixel and broadcast across the years.
pixels_with_covariates = attach_covariates(pixels, covariates, aoi_nc_peat_80_histosol, res_m)

print(pixels_with_covariates.shape)
print("treated pixel-years by calendar year:")
print(pixels_with_covariates.groupby("year")["treated"].sum())
pixels_with_covariates.head()

## 2b. Event-time panel → staggered DiD

The pixel-year panel above already carries what the staggered
difference-in-differences in `peatfire.modeling.did` needs: a per-pixel
`treated` that switches on at each site's `restoration_year`, and that
restoration year itself (the Callaway–Sant'Anna **cohort** `g`). Below we
attach the fire response per year, give every pixel a stable `unit_id`, thread
`restoration_year` into `did.attach_cohort`, and fit the group-time ATT.

This is the design that identifies off the *change* in burning after each
site's restoration relative to not-yet/never-restored controls, so every
time-invariant confounder differences out. The `build_frame` →
`fit_logit_clustered` route in §3–4 is the **levels / odds-ratio** alternative,
which consumes matched `units` polygons from Stage 5–7 rather than this panel.

In [ ]:
# --- 1. fire response per (pixel, year), sampled from the swappable product ---
# Mirrors build_frame's per-year response step, but samples at pixel points
# instead of rasterizing polygons. Needs the fire product on disk.
product = "FireCCIS311"
grid = build_common_grid(aoi_nc_peat_80_histosol, res_m=res_m)

panel = pixels_with_covariates.copy()
panel["burned"] = np.nan
for year, idx in panel.groupby("year").groups.items():
    resp = load_standardized(product, int(year), aoi_nc_peat_80_histosol)
    if resp is None:                       # product missing this year -> leave NaN
        continue
    burned = to_common_grid(resp.astype("float32"), grid, how="max")
    sub = panel.loc[idx]
    xi = xr.DataArray(sub["x"].values, dims="point")
    yi = xr.DataArray(sub["y"].values, dims="point")
    panel.loc[idx, "burned"] = burned.sel(x=xi, y=yi, method="nearest").values

# --- 2. one stable entity id per distinct pixel (the DiD panel's entity) ---
panel["unit_id"] = panel.groupby(["x", "y"]).ngroup()

# --- 3. thread restoration_year -> CS cohort `g` (0 = never-treated controls) ---
# cohort_by maps each site to its first-treatment (restoration) year; control-pool
# pixels have no Proj_Name and fall through to g=0 (never-treated).
cohort_by = (
    panel.dropna(subset=["restoration_year"])
    .groupby("Proj_Name")["restoration_year"]
    .first()
)
panel = did.attach_cohort(panel, cohort_by=cohort_by, key="Proj_Name", treated_col="treated")

print("cohorts g:", sorted(panel["g"].unique()))
print("burned pixel-years with coverage:", int(panel["burned"].notna().sum()))
panel.head()

In [ ]:
# Reshape to the balanced (unit, year) panel and fit the doubly-robust
# group-time ATT (Callaway & Sant'Anna 2021). Needs `pip install differences`
# and a `burned` column with coverage, so we drop the year-loop's NaNs first.
covs = [c for c in ("elevation", "histosol_pct") if c in panel.columns]

cs_panel = did.build_panel(
    panel.dropna(subset=["burned"]),
    entity="unit_id",
    time="year",
    response="burned",
    covariates=covs,
)

att = did.estimate_att(cs_panel, response="burned", covariates=covs, cluster="unit_id")

overall = did.aggregate_att(att, kind="simple")   # Castro's headline ATT
event_study = did.aggregate_att(att, kind="event")  # pre-trends check + dynamics
print("overall ATT:", overall)
event_study

In [6]:
pixels_with_covariates

,x,y,geometry,treated,histosol_pct,elevation
0,1.721560e+06,1.674276e+06,POINT (1721560.262 1674275.871),1,90.0,16.383682
1,1.721860e+06,1.674276e+06,POINT (1721860.262 1674275.871),1,90.0,14.711100
2,1.722160e+06,1.674276e+06,POINT (1722160.262 1674275.871),1,90.0,11.696195
3,1.722460e+06,1.674276e+06,POINT (1722460.262 1674275.871),1,90.0,6.692596
4,1.722760e+06,1.674276e+06,POINT (1722760.262 1674275.871),1,90.0,12.696268
...,...,...,...,...,...,...
56371,1.594660e+06,1.353576e+06,POINT (1594660.262 1353575.871),0,NaN,NaN
56372,1.594360e+06,1.352676e+06,POINT (1594360.262 1352675.871),0,NaN,NaN
56373,1.594660e+06,1.352676e+06,POINT (1594660.262 1352675.871),0,NaN,NaN
56374,1.596460e+06,1.352676e+06,POINT (1596460.262 1352675.871),0,NaN,NaN


In [10]:
import rioxarray
x = rioxarray.open_rasterio(data_path('processed', 'topography', 'GLO30_DEM', 'GLO30_DEM_nc.tif'), masked=True)
x = x.rio.reproject('EPSG:5070')
x.plot()

: 

### Balance plot (before / after matching)

The figure that *justifies* the design: overlaid distributions of elevation and
histosol % for treated vs control, before and after matching. Before, the groups
are offset; after, they should overlap -- that overlap is what lets you later
attribute a fire difference to restoration rather than geography. The
controls-next-to-restoration *map* is just a sanity check.

In [7]:
# TODO(you): 2x2 (or 1x2) of elevation / histosol_pct, treated vs control,
# pre- and post-match. e.g. seaborn.kdeplot or overlaid hist per covariate.
# Use peatfire.set_fire_style() for consistent styling.

## 3. Build the tidy pixel-year frame

In [9]:
# Consumes the matched `units`. Response is swappable via `product=`.
frame = build_frame(
    units,
    product="FireCCIS311",   # swap for a severity product to model severity
    years=range(2019, 2025),
    covariate_names=None,    # default = every covariate on disk (elev, histosol),
    site_id_col='Proj_Name',
    
)
print(frame.shape)
print("burn rate by treatment:")
print(frame.groupby("treated")["burned"].mean())
frame.head()

NameError: name 'units' is not defined

## 4. Fit + odds ratios

In [ ]:
# Cluster-robust logistic: honest SEs (clustered on site_id), not one-per-pixel.
covs = [c for c in ("elevation", "histosol_pct") if c in frame.columns]
result = fit_logit_clustered(frame, covariates=covs)   # burned ~ treated + covs
print(result.summary())

# The headline: exp(beta) with CIs. treated < 1 => restoration lowers fire odds.
or_table = odds_ratios(result)
or_table

In [ ]:
# Prettier table for the slides (decisions/roadmap note): colour the odds ratios.
(or_table.style
    .format({"beta": "{:.3f}", "odds_ratio": "{:.2f}",
             "or_ci_low": "{:.2f}", "or_ci_high": "{:.2f}", "p_value": "{:.3f}"})
    .background_gradient(subset=["odds_ratio"], cmap="RdBu_r", vmin=0, vmax=2))

### Next

- Add a `treated:precip` interaction (dry-year effect) once climate is downloaded:
  pass `formula="burned ~ treated * precip + elevation + histosol_pct"`.
- Step up to `fit_mixed_logit` (site random intercept) to cross-check the SEs.
- Swap `product=` to a severity layer to re-run the whole thing for severity.